## Stochastycznna metoda gradietnu

In [18]:
import numpy as np
import time
np.random.seed(42)
x = 2  * np.random.rand(100, 1)
y = 4 + 3 * x + np.random.rand(100, 1)

x_b = np.c_[np.ones((100, 1)), x]   # bias term


### Prosta regresja za pomoca stochastycznego gradientu

In [26]:
def SGD(X, y, epochs=50, lr=0.01):
    m = len(x)  # liczba danych
    theta = np.random.rand(2, 1)  # losowy parametr  ma rozmiac 2x1 poniewaz jest to prosta regresja 
    for ep in range(epochs):
        indicies = np.random.permutation(m)  # losowa permutacja indeksow danych
        x_shuffled = x_b[indicies]
        y_shuffled = y[indicies]
        for i in indicies:
            xi = x_shuffled[i:i + 1]
            yi = y_shuffled[i:i + 1]
            grad = 2 * xi.T.dot(xi.dot(theta) - yi) # liczymy gradient
            theta = theta - lr * grad  # aktualizujemy paramter
    return theta

time_start = time.perf_counter()
opt_theta = SGD(x_b, y, epochs=100, lr=0.01)
time_end = time.perf_counter()
time_numpy = time_end - time_start
print(f"Czas wykonania: {time_numpy:.6f} sekund")
y_pred = x_b.dot(opt_theta)
rss = np.sum((y - y_pred) ** 2)
tss = np.sum((y - np.mean(y)) ** 2)
r2_numpy = 1 - rss / tss
print(f"Optymalny parametr: {opt_theta.ravel()}, RSS: {rss}, R²: {r2_numpy}")

Czas wykonania: 0.026681 sekund
Optymalny parametr: [4.52487607 2.98124863], RSS: 8.50465064227666, R²: 0.9734556394555335


##### Teraz zrobimy przy uzyciu sickit learn

In [21]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score

model  = SGDRegressor(
    max_iter=1000,
    tol=1e-4,
    random_state=42 ,
    eta0=0.01,    # learning rate  od eta greckiego
    learning_rate='constant'  # utrzymuje parametr w czasie staly
    )

time_start = time.perf_counter()
model.fit(x, y.ravel())
time_end = time.perf_counter()
time_sklearn = time_end - time_start
print(f"Czas wykonania: {time_sklearn:.6f} sekund")

bias =  model.intercept_[0]   # zapisujemy wynik regresdji w zmiennych indeksujemy zevby dostac sama wartosc
coeff = model.coef_[0]

y_pred = model.predict(x)
r2 = r2_score(y, y_pred)
print(f"Scikit-Learn Bias (wyraz wolny): {bias:.3f}")
print(f"Scikit-Learn Weight (waga):     {coeff:.3f}")
print(f"Współczynnik R^2:                {r2:.4f}")

Czas wykonania: 0.000813 sekund
Scikit-Learn Bias (wyraz wolny): 4.498
Scikit-Learn Weight (waga):     2.998
Współczynnik R^2:                0.9735


##### Roznica czasow algorytmow

In [23]:
print(f'Rożnica w czasie wykonania: {time_numpy - time_sklearn:.6f} sekund w zgledem numpy')
print(f'iloraz w czasie wykonania: {time_numpy / time_sklearn:.6f} sekund w zgledem sklearn')

Rożnica w czasie wykonania: 0.026453 sekund w zgledem numpy
iloraz w czasie wykonania: 33.550018 sekund w zgledem sklearn


### Teraz porownajmy prosty gradient

In [36]:
def simple_gradient(x,y, lr = 0.01, iterations = 100, tol = 1e-6):
    m = len(x)
    theta = np.random.rand(2, 1)
    for i in range(iterations):
        grad = 2 * x.T.dot(x.dot(theta) - y)
        theta = theta - lr * grad
        if np.linalg.norm(grad) < tol:
            break
    return theta
    
time_start = time.perf_counter()
theta_simple = simple_gradient(x_b, y, iterations=500)
time_end = time.perf_counter()
time_simple = time_end - time_start
print(f"Simple Gradient Descent - Czas wykonania: {time_simple:.6f} sekund")

rss_simple = np.sum((y - x_b.dot(theta_simple)) ** 2)
tss_simple = np.sum((y - np.mean(y)) ** 2)
r2_simple = 1 - rss_simple / tss_simple
print(f"Simple Gradient Descent - Optymalny parametr: {theta_simple.ravel()}, RSS: {rss_simple}, R²: {r2_simple}")

Simple Gradient Descent - Czas wykonania: 0.003118 sekund
Simple Gradient Descent - Optymalny parametr: [-1.77148778e+248 -2.00633843e+248], RSS: inf, R²: -inf


c:\Users\filip\miniconda3\Lib\site-packages\numpy\linalg\_linalg.py:2784: RuntimeWarning: overflow encountered in dot
  sqnorm = x.dot(x)
C:\Users\filip\AppData\Local\Temp\ipykernel_20160\2241820657.py:17: RuntimeWarning: overflow encountered in square
  rss_simple = np.sum((y - x_b.dot(theta_simple)) ** 2)


##### Porownamy czasy i wyniki stochastycznego gradientu i gradientu prostego

In [27]:
print(f'Roznica w R2: {r2_simple - r2_numpy:.6f}')
print(f'Roznica w czasie wykonania: {time_simple - time_numpy:.6f} sekund w zgledem simple gradient')
print(f'Iloraz w czasie wykonania: {time_simple / time_numpy:.6f} sekund w zgledem simple gradient')

Roznica w R2: -161.421233
Roznica w czasie wykonania: -0.024534 sekund w zgledem simple gradient
Iloraz w czasie wykonania: 0.080441 sekund w zgledem simple gradient
